# General setup

In [1]:
# Imports
import matplotlib
matplotlib.use('Agg')
import argparse
import re
import json
import warnings
import numpy as np
from modules.CHILI import CHILI
from modules.net import SCVAE
from torch_geometric.loader import DataLoader
import torch
from torch.utils.data import TensorDataset
import datetime
import pathlib
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from ase import Atoms
from ase.io import write
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from modules.loss_functions import weighted_MSELoss, weighted_CrossEntropyLoss
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")

In [2]:
# Functions
def create_cif(cell_params, cell_positions, cell_atoms, filename, prediction=True, composition=None, simplified_atom_identities=False):
    """
    Create a CIF file from the cell parameters, positions and atoms
    """
    if prediction:
        # Find argmax of atoms
        cell_atoms = np.argmax(cell_atoms, axis=1)

    # Remove atoms with atom number 0
    cell_positions = cell_positions[cell_atoms != 0]
    cell_atoms = cell_atoms[cell_atoms != 0]
    
    # Remove atoms not in the unit cell
    cell_atoms = cell_atoms[(cell_positions[:,0] < 0.95) & (cell_positions[:,1] < 0.95) & (cell_positions[:,2] < 0.95)]
    cell_positions = cell_positions[(cell_positions[:,0] < 0.95) & (cell_positions[:,1] < 0.95) & (cell_positions[:,2] < 0.95)]
    
    
    if simplified_atom_identities:
        cell_atoms = np.where(cell_atoms == 1, 8, cell_atoms)
        cell_atoms = np.where(cell_atoms == 2, 26, cell_atoms)
    
    # Create Atoms object
    atoms = Atoms(cell_atoms, scaled_positions=cell_positions, cell=cell_params)

    if not composition:
        composition = str(atoms.symbols)

    # Write CIF
    write(filename + f'.cif', images=atoms, format='cif') # _{composition}

    if not prediction:
        return composition
    return None

In [3]:
# Setup
model_path = './models/Interpolation_Supercell_latentLog_beta_annealing_2d_latentMSE/'
model_name = 'best_model.pth' # 'best_model_annealed.pth'
setup_json_path = f'{model_path}setup_json.json'
with open(setup_json_path, 'r') as setup_json_file:
    setup_json = json.load(setup_json_file)

# Make prediction folders
experimental_folder = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/experimental_predictions'
pathlib.Path(experimental_folder).mkdir(parents=True, exist_ok=True)

interpolation_folder = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/interpolation_predictions'
pathlib.Path(interpolation_folder).mkdir(parents=True, exist_ok=True)

sample_folder = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/sample_predictions'
pathlib.Path(sample_folder).mkdir(parents=True, exist_ok=True)

# Make paper figures folder
paper_figures_folder = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures'
pathlib.Path(paper_figures_folder).mkdir(parents=True, exist_ok=True)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
# Load model
model = SCVAE(
    latent_dim=setup_json['model']['latent_dim'],
    out_dim=setup_json['model']['out_dim'],
    prior_factor=setup_json['model']['prior_factor'],
    gnn_dim=setup_json['model']['gnn_dim'],
    gnn_heads=setup_json['model']['gnn_heads'],
    gnn_edge_dim=setup_json['model']['gnn_edge_dim'],
    scattering_channels=setup_json['model']['scattering_channels'],
    scattering_dim=setup_json['model']['scattering_dim'],
    scattering_kernel_size=setup_json['model']['scattering_kernel_size'],
    scattering_stride=setup_json['model']['scattering_stride'],
    scattering_padding=setup_json['model']['scattering_padding'],
    composition_dim=setup_json['model']['composition_dim'],
    decoder_hidden_dim=setup_json['model']['decoder_hidden_dim'],
    position_output_dim=setup_json['model']['position_output_dim'],
    atom_output_dim=setup_json['model']['atom_output_dim'],
    cell_output_dim=setup_json['model']['cell_output_dim'],
    seperate_decoder=setup_json['training']['seperate_decoder'],
).to(device)

# Load model weights
model.load_state_dict(torch.load(model_path + model_name))

<All keys matched successfully>

In [5]:
# Load normalization parameters
if setup_json['data']['normalize_cell_parameters']:
    cell_means = torch.tensor([
        setup_json['data']['cell_normalization']['a']['mean'],
        setup_json['data']['cell_normalization']['b']['mean'],
        setup_json['data']['cell_normalization']['c']['mean'],
        setup_json['data']['cell_normalization']['alpha']['mean'],
        setup_json['data']['cell_normalization']['beta']['mean'],
        setup_json['data']['cell_normalization']['gamma']['mean'],
    ]).float().to(device)
    cell_stds = torch.tensor([
        setup_json['data']['cell_normalization']['a']['std'],
        setup_json['data']['cell_normalization']['b']['std'],
        setup_json['data']['cell_normalization']['c']['std'],
        setup_json['data']['cell_normalization']['alpha']['std'],
        setup_json['data']['cell_normalization']['beta']['std'],
        setup_json['data']['cell_normalization']['gamma']['std'],
    ]).float().to(device)

if setup_json['data']['normalize_atom_positions']:
    atom_position_means = torch.tensor(setup_json['data']['atom_position_normalization']['mean']).float().to(device)
    atom_position_stds = torch.tensor(setup_json['data']['atom_position_normalization']['std']).float().to(device)

if setup_json['data']['normalize_distances']:
    distance_means = torch.tensor(setup_json['data']['distance_normalization']['mean']).float().to(device)
    distance_stds = torch.tensor(setup_json['data']['distance_normalization']['std']).float().to(device)

beta = setup_json['training']['beta']
out_dim = setup_json['model']['out_dim']

# Load model test results

## Code

In [6]:
# Load results from test script
with open(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/CrystalTypeAnalysis/losses_{model_name[:-4]}.json', 'r') as f:
    losses_json = json.load(f)
df_losses = pd.DataFrame(losses_json)

with open(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/CrystalTypeAnalysis/reconstructions_{model_name[:-4]}.json', 'r') as f:
    rec_json = json.load(f)
df_rec = pd.DataFrame(rec_json)
# Capitalize crystal types if not already
name_map = {'interpolated': 'Interpolated', 'rocksalt': 'RockSalt', 'spinel': 'Spinel', 'zincblende': 'ZincBlende'}
df_rec['crystalType'] = df_rec['crystalType'].apply(lambda x: name_map[x] if x in name_map else x)

if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    df_rec[['ls1', 'ls2']] = df_rec['latent_space_mean'].apply(pd.Series)
else:
    # Perform PCA
    pca = PCA(n_components=2)
    pca.fit(np.array(df_rec['latent_space_mean'].values.tolist()))
    df_rec[['pc1', 'pc2']] = pca.transform(np.array(df_rec['latent_space_mean'].values.tolist()))

In [7]:
df_losses.head()

,total,reconstruction_loss,cell_parameters,cell_positions,cell_atoms,kld,crystalType,particleSize
0,-0.856868,0.248447,0.001420,0.067893,0.179134,0.170812,interpolated,54.079006
1,-0.649614,0.330071,0.000079,0.083879,0.246112,0.191177,interpolated,11.524527
2,-0.598401,0.407590,0.000474,0.075244,0.331872,0.141300,interpolated,23.376171
3,-0.504929,0.535124,0.000781,0.084206,0.450137,0.068174,interpolated,33.738770
4,-0.846443,0.162169,0.012779,0.068686,0.080705,0.250922,interpolated,23.986713


In [8]:
df_rec.head()

,crystalType,n_atoms,n_oxygens,n_metals,cell_parameters,cell_positions,cell_atoms,latent_space_mean,latent_space_std,true_cell_parameters,true_cell_positions,true_cell_atoms,ls1,ls2
0,Interpolated,64,33,31,"[8.59115982055664, 8.593923568725586, 8.583761...","[[0.03798000141978264, -0.018060000613331795, ...","[1, 2, 2, 2, 1, 1, 1, 2, 1, 1, 1, 2, 2, 2, 2, ...","[1.3172607421875, 0.6281828880310059]","[0.08410815894603729, 0.020766206085681915]","[-0.8238851428031921, -0.8238851428031921, -0....","[[0.0049600000493228436, 0.0049600000493228436...","[8, 32, 32, 32, 8, 8, 8, 32, 8, 8, 8, 32, 32, ...",1.317261,0.628183
1,Interpolated,64,41,23,"[8.826685905456543, 8.825448036193848, 8.82950...","[[0.0101500004529953, 0.020109999924898148, 0....","[1, 2, 1, 1, 1, 1, 1, 1, 2, 2, 1, 1, 1, 2, 2, ...","[0.502522349357605, 0.3269649147987366]","[0.03658655658364296, 0.012121125124394894]","[-0.6218971610069275, -0.6218971610069275, -0....","[[0.0024800000246614218, 0.0024800000246614218...","[8, 30, 8, 8, 8, 8, 8, 8, 30, 30, 8, 8, 8, 30,...",0.502522,0.326965
2,Interpolated,64,37,27,"[8.82210922241211, 8.825701713562012, 8.826155...","[[0.023240000009536743, 0.007499999832361937, ...","[1, 2, 2, 2, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 2, ...","[0.5169130563735962, 0.027201466262340546]","[0.038836006075143814, 0.012316962704062462]","[-0.6097007989883423, -0.6097007989883423, -0....","[[0.01735999993979931, 0.01735999993979931, 0....","[8, 73, 73, 8, 8, 8, 73, 8, 8, 8, 8, 8, 8, 73,...",0.516913,0.027201
3,Interpolated,64,44,20,"[8.642810821533203, 8.633030891418457, 8.62706...","[[0.018559999763965607, -0.011380000039935112,...","[1, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, ...","[0.650273859500885, 0.10349880158901215]","[0.039961837232112885, 0.013337472453713417]","[-0.790813684463501, -0.790813684463501, -0.79...","[[0.014879999682307243, 0.014879999682307243, ...","[8, 76, 8, 8, 8, 76, 8, 8, 8, 8, 8, 8, 76, 76,...",0.650274,0.103499
4,Interpolated,64,32,32,"[8.93780517578125, 8.956751823425293, 8.942300...","[[0.03172999992966652, 0.016359999775886536, 0...","[1, 2, 2, 2, 1, 1, 1, 2, 1, 1, 1, 2, 2, 2, 2, ...","[0.2824069559574127, 0.6403026580810547]","[0.04450659826397896, 0.016283512115478516]","[-0.3556883931159973, -0.3556883931159973, -0....","[[0.0024800000246614218, 0.0024800000246614218...","[8, 52, 52, 52, 8, 8, 8, 52, 8, 8, 8, 52, 52, ...",0.282407,0.640303


In [9]:
df_combined = pd.concat([df_losses.drop('crystalType', axis=1), df_rec], axis=1)
df_combined.head()

,total,reconstruction_loss,cell_parameters,cell_positions,cell_atoms,kld,particleSize,crystalType,n_atoms,n_oxygens,...,cell_parameters,cell_positions,cell_atoms,latent_space_mean,latent_space_std,true_cell_parameters,true_cell_positions,true_cell_atoms,ls1,ls2
0,-0.856868,0.248447,0.001420,0.067893,0.179134,0.170812,54.079006,Interpolated,64,33,...,"[8.59115982055664, 8.593923568725586, 8.583761...","[[0.03798000141978264, -0.018060000613331795, ...","[1, 2, 2, 2, 1, 1, 1, 2, 1, 1, 1, 2, 2, 2, 2, ...","[1.3172607421875, 0.6281828880310059]","[0.08410815894603729, 0.020766206085681915]","[-0.8238851428031921, -0.8238851428031921, -0....","[[0.0049600000493228436, 0.0049600000493228436...","[8, 32, 32, 32, 8, 8, 8, 32, 8, 8, 8, 32, 32, ...",1.317261,0.628183
1,-0.649614,0.330071,0.000079,0.083879,0.246112,0.191177,11.524527,Interpolated,64,41,...,"[8.826685905456543, 8.825448036193848, 8.82950...","[[0.0101500004529953, 0.020109999924898148, 0....","[1, 2, 1, 1, 1, 1, 1, 1, 2, 2, 1, 1, 1, 2, 2, ...","[0.502522349357605, 0.3269649147987366]","[0.03658655658364296, 0.012121125124394894]","[-0.6218971610069275, -0.6218971610069275, -0....","[[0.0024800000246614218, 0.0024800000246614218...","[8, 30, 8, 8, 8, 8, 8, 8, 30, 30, 8, 8, 8, 30,...",0.502522,0.326965
2,-0.598401,0.407590,0.000474,0.075244,0.331872,0.141300,23.376171,Interpolated,64,37,...,"[8.82210922241211, 8.825701713562012, 8.826155...","[[0.023240000009536743, 0.007499999832361937, ...","[1, 2, 2, 2, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 2, ...","[0.5169130563735962, 0.027201466262340546]","[0.038836006075143814, 0.012316962704062462]","[-0.6097007989883423, -0.6097007989883423, -0....","[[0.01735999993979931, 0.01735999993979931, 0....","[8, 73, 73, 8, 8, 8, 73, 8, 8, 8, 8, 8, 8, 73,...",0.516913,0.027201
3,-0.504929,0.535124,0.000781,0.084206,0.450137,0.068174,33.738770,Interpolated,64,44,...,"[8.642810821533203, 8.633030891418457, 8.62706...","[[0.018559999763965607, -0.011380000039935112,...","[1, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, ...","[0.650273859500885, 0.10349880158901215]","[0.039961837232112885, 0.013337472453713417]","[-0.790813684463501, -0.790813684463501, -0.79...","[[0.014879999682307243, 0.014879999682307243, ...","[8, 76, 8, 8, 8, 76, 8, 8, 8, 8, 8, 8, 76, 76,...",0.650274,0.103499
4,-0.846443,0.162169,0.012779,0.068686,0.080705,0.250922,23.986713,Interpolated,64,32,...,"[8.93780517578125, 8.956751823425293, 8.942300...","[[0.03172999992966652, 0.016359999775886536, 0...","[1, 2, 2, 2, 1, 1, 1, 2, 1, 1, 1, 2, 2, 2, 2, ...","[0.2824069559574127, 0.6403026580810547]","[0.04450659826397896, 0.016283512115478516]","[-0.3556883931159973, -0.3556883931159973, -0....","[[0.0024800000246614218, 0.0024800000246614218...","[8, 52, 52, 52, 8, 8, 8, 52, 8, 8, 8, 52, 52, ...",0.282407,0.640303


In [10]:
# Load latent log if it exists
import yaml
try:
    df_latent_log = pd.read_csv(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/latent_space_log.csv')

    df_latent_log = df_latent_log.drop(df_latent_log[df_latent_log['posterior_mean'] == 'posterior_mean'].index)

    # df_latent_log = df_latent_log[:1000]

    # Select only every tenth epoch
    # Convert epoch to ints
    df_latent_log['epoch'] = df_latent_log['epoch'].astype(np.int16)

    df_latent_log = df_latent_log[df_latent_log['epoch'] % 10 == 0]

    df_latent_log[['posterior_mean', 'posterior_std', 'prior_mean', 'prior_std']] = df_latent_log[['posterior_mean', 'posterior_std', 'prior_mean', 'prior_std']].applymap(yaml.safe_load)

    if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
        df_latent_log[['ls1', 'ls2']] = df_latent_log['posterior_mean'].apply(pd.Series)
        df_latent_log[['ls1_prior', 'ls2_prior']] = df_latent_log['prior_mean'].apply(pd.Series)
    else:
        # Perform PCA
        # pca = PCA(n_components=2)
        # pca.fit(np.array(df_latent_log['posterior_mean'].values.tolist()))
        df_latent_log[['pc1', 'pc2']] = pca.transform(np.array(df_latent_log['posterior_mean'].values.tolist()))

        df_latent_log[['pc1_prior', 'pc2_prior']] = pca.transform(np.array(df_latent_log['prior_mean'].values.tolist()))
        
    print(df_latent_log.columns)
except:
    df_latent_log = None
    print('No latent log found')

Index(['epoch', 'posterior_mean', 'posterior_std', 'prior_mean', 'prior_std',
       'np_size', 'crystal_type', 'crystal_system', 'space_group', 'ls1',
       'ls2', 'ls1_prior', 'ls2_prior'],
      dtype='object')


## Plot

In [11]:
# Set seaborn style
# sns.set_theme(style='darkgrid', font_scale=1.)
# Make animation of latent space throughout 
# Plot latent space
if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df_rec, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df_rec, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_means.pdf', dpi=300)

In [12]:
# Plot latent space with surface showing the loss
from scipy.interpolate import griddata
import matplotlib.colors as colors

# Decide on contour column
contour_column = 'reconstruction_loss'

# Get min and max values
min_loss = df_combined[contour_column].min()
max_loss = df_combined[contour_column].max()

# Plot
if len(df_combined['latent_space_mean'].iloc[0]) == 2:
    # Interpolate z values to a grid from points
    xi = np.linspace(df_combined['ls1'].min()*1.1, df_combined['ls1'].max()*1.1, 1000)
    yi = np.linspace(df_combined['ls2'].min()*1.1, df_combined['ls2'].max()*1.1, 1000)
    zi = griddata((df_combined['ls1'], df_combined['ls2']), df_combined[contour_column], (xi[None,:], yi[:,None]), method='linear', fill_value=np.nan)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xi, yi, zi, 100, cmap='viridis')#, norm=colors.LogNorm())
    sns.scatterplot(data=df_combined, x='ls1', y='ls2', hue='crystalType', style='crystalType', palette='tab20', s=50)
    plt.colorbar()
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    
    # Interpolate z values to a grid from points
    xi = np.linspace(df_combined['pc1'].min()*1.1, df_combined['pc1'].max()*1.1, 1000)
    yi = np.linspace(df_combined['pc2'].min()*1.1, df_combined['pc2'].max()*1.1, 1000)
    zi = griddata((df_combined['pc1'], df_combined['pc2']), df_combined[contour_column], (xi[None,:], yi[:,None]), method='linear', fill_value=np.nan)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xi, yi, zi, 100, cmap='viridis')#, norm=colors.LogNorm())
    sns.scatterplot(data=df_combined, x='pc1', y='pc2', hue='crystalType', style='crystalType', palette='tab20', s=50)
    plt.colorbar()
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()


In [13]:
# # 2D scatter plot with slider

# symbols = ['circle', 'x', 'square', 'diamond', 'cross', 'triangle-up', 'triangle-down', 'star', 'hexagon', 'pentagon', 'hexagram', 'octagon', 'star-triangle-up', 'star-triangle-down', 'star-square', 'star-diamond', 'star-cross', 'hourglass', 'bowtie', 'circle-cross', 'circle-x', 'circle-star', 'circle-triangle-up', 'circle-triangle-down', 'circle-diamond', 'cross-thin', 'x-thin', 'square-thin', 'diamond-thin', 'cross-dot', 'x-dot', 'square-dot', 'diamond-dot', 'cross-open', 'x-open', 'square-open', 'diamond-open', 'cross-thin-open', 'x-thin-open', 'square-thin-open', 'diamond-thin-open', 'pentagon-open', 'hexagon-open', 'hexagram-open', 'octagon-open', 'star-open', 'star-triangle-up-open', 'star-triangle-down-open', 'star-square-open', 'star-diamond-open', 'star-cross-open', 'hourglass-open', 'bowtie-open', 'circle-cross-open', 'circle-x-open', 'circle-star-open', 'circle-triangle-up-open', 'circle-triangle-down-open', 'circle-diamond-open']
# color_scale = px.colors.qualitative.Dark24

# if df_latent_log is not None:
#     crystal_types = df_latent_log['crystal_type'].unique()
#     n_crystal_types = len(crystal_types)
#     trace_list = []
#     if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
#         pass
#     else:
#         for epoch in df_latent_log['epoch'].unique():
#             df_epoch = df_latent_log[df_latent_log['epoch'] == epoch]
#             for i, crystal_type in enumerate(crystal_types):
#                 df_crystal = df_epoch[df_epoch['crystal_type'] == crystal_type]
#                 trace = go.Scatter(
#                     x=df_crystal['pc1'],
#                     y=df_crystal['pc2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         size=10,
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                     ),
#                     visible=False,
#                 )
#                 trace_list.append(trace)
        
#         fig = go.Figure(data=trace_list)
        
#         for i in range(n_crystal_types):
#             fig.data[i]['visible'] = True
        
#         # Add slider
#         steps = []
#         for i, epoch in enumerate(df_latent_log['epoch'].unique()):
#             step = dict(
#                 method='restyle',
#                 args=['visible', [False] * len(trace_list)],
#                 label=f'{epoch}',
#             )
#             for j, crystal_type in enumerate(crystal_types):
#                 step['args'][1][i*n_crystal_types + j] = True
#             steps.append(step)    
        
#         sliders = [dict(
#             active=0,
#             currentvalue={"prefix": "Epoch: "},
#             pad={"t": 50},
#             steps=steps,
#         )]
        
#         fig.update_layout(
#             sliders=sliders,
#             width=1200,
#             height=800,
#             )
        
#         # Set axis labels
#         fig.update_xaxes(title_text='PC 1')
#         fig.update_yaxes(title_text='PC 2')
        
#         fig.show()

In [14]:
# Sample n points from each distribution in the latent space log
if df_latent_log is not None:
    n_samples = 100
    
    crystal_type_list = []
    latent_space_list = []
    epoch_list = []
    for latent_mean, latent_std, crystal_type, epoch in zip(df_latent_log['posterior_mean'], df_latent_log['posterior_std'], df_latent_log['crystal_type'], df_latent_log['epoch']):
        latent_space_samples = torch.distributions.Normal(loc=torch.tensor(latent_mean), scale=torch.tensor(latent_std)).sample((n_samples,)).numpy()
        latent_space_list.extend(latent_space_samples)
        crystal_type_list.extend([crystal_type] * n_samples)
        epoch_list.extend([epoch] * n_samples)

    if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
        df_log_samples = pd.DataFrame(latent_space_list, columns=['ls1', 'ls2'])
    else:
        latent_space_list = pca.transform(latent_space_list)
        df_log_samples = pd.DataFrame(latent_space_list, columns=['pc1', 'pc2'])

    df_log_samples['crystalType'] = crystal_type_list
    df_log_samples['epoch'] = epoch_list

In [15]:
# # 2D scatter plot and 2D histogram with slider

# symbols = ['circle', 'x', 'square', 'diamond', 'cross', 'triangle-up', 'triangle-down', 'star', 'hexagon', 'pentagon', 'hexagram', 'octagon', 'star-triangle-up', 'star-triangle-down', 'star-square', 'star-diamond', 'star-cross', 'hourglass', 'bowtie', 'circle-cross', 'circle-x', 'circle-star', 'circle-triangle-up', 'circle-triangle-down', 'circle-diamond', 'cross-thin', 'x-thin', 'square-thin', 'diamond-thin', 'cross-dot', 'x-dot', 'square-dot', 'diamond-dot', 'cross-open', 'x-open', 'square-open', 'diamond-open', 'cross-thin-open', 'x-thin-open', 'square-thin-open', 'diamond-thin-open', 'pentagon-open', 'hexagon-open', 'hexagram-open', 'octagon-open', 'star-open', 'star-triangle-up-open', 'star-triangle-down-open', 'star-square-open', 'star-diamond-open', 'star-cross-open', 'hourglass-open', 'bowtie-open', 'circle-cross-open', 'circle-x-open', 'circle-star-open', 'circle-triangle-up-open', 'circle-triangle-down-open', 'circle-diamond-open']
# color_scale = px.colors.qualitative.Dark24

# if df_latent_log is not None:
#     crystal_types = df_latent_log['crystal_type'].unique()
#     n_crystal_types = len(crystal_types)
#     trace_list = []
#     if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
#         for epoch in df_latent_log['epoch'].unique():
#             trace_means = []
#             df_epoch = df_latent_log[df_latent_log['epoch'] == epoch]
#             df_epoch_samples = df_log_samples[df_log_samples['epoch'] == epoch]
#             for i, crystal_type in enumerate(crystal_types):
#                 df_crystal = df_epoch[df_epoch['crystal_type'] == crystal_type]
#                 df_crystal_samples = df_epoch_samples[df_epoch_samples['crystalType'] == crystal_type]
#                 trace = go.Scatter(
#                     x=df_crystal['ls1'],
#                     y=df_crystal['ls2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         size=10,
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                         line=dict(
#                             color='black',
#                             width=1,
#                         ),
#                     ),
#                     visible=False,
#                 )
                
                
#                 # Sampled points
#                 trace_samples = go.Scatter(
#                     x=df_crystal_samples['ls1'],
#                     y=df_crystal_samples['ls2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                         opacity=0.5,
#                     ),
#                     visible=False,
#                 )
#                 trace_list.append(trace_samples)
#                 trace_means.append(trace)
#             trace_list.extend(trace_means)
            
#         fig = go.Figure(data=trace_list)
        
#         for i in range(n_crystal_types*2):
#             fig.data[i]['visible'] = True
            
#         n_steps = len(df_latent_log['epoch'].unique())
        
#         # Add slider
#         steps = []
#         for i, epoch in enumerate(df_latent_log['epoch'].unique()):
#             step = dict(
#                 method='restyle',
#                 args=['visible', [False] * len(trace_list)],
#                 label=f'{epoch}',
#             )
#             for j in range(n_crystal_types*2):
#                 step['args'][1][i*n_crystal_types*2 + j] = True
                
#             steps.append(step)
            
#         sliders = [dict(
#             active=0,
#             currentvalue={"prefix": "Epoch: "},
#             pad={"t": 50},
#             steps=steps,
#         )]
        
#         fig.update_layout(
#             sliders=sliders,
#             width=1200,
#             height=800,
#             )
        
#         # Set axis labels
#         fig.update_xaxes(title_text='LS dim 1')
#         fig.update_yaxes(title_text='LS dim 2')
        
#         fig.write_html(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/latent_space_samples.html')
#     else:
#         for epoch in df_latent_log['epoch'].unique():
#             trace_means = []
#             df_epoch = df_latent_log[df_latent_log['epoch'] == epoch]
#             df_epoch_samples = df_log_samples[df_log_samples['epoch'] == epoch]
#             for i, crystal_type in enumerate(crystal_types):
#                 df_crystal = df_epoch[df_epoch['crystal_type'] == crystal_type]
#                 df_crystal_samples = df_epoch_samples[df_epoch_samples['crystalType'] == crystal_type]
#                 trace = go.Scatter(
#                     x=df_crystal['pc1'],
#                     y=df_crystal['pc2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         size=10,
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                         line=dict(
#                             color='black',
#                             width=1,
#                         ),
#                     ),
#                     visible=False,
#                 )
                
                
#                 # Sampled points
#                 trace_samples = go.Scatter(
#                     x=df_crystal_samples['pc1'],
#                     y=df_crystal_samples['pc2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                         opacity=0.5,
#                     ),
#                     visible=False,
#                 )
#                 trace_list.append(trace_samples)
#                 trace_means.append(trace)
#             trace_list.extend(trace_means)
            
#         fig = go.Figure(data=trace_list)
        
#         for i in range(n_crystal_types*2):
#             fig.data[i]['visible'] = True
        
#         n_steps = len(df_latent_log['epoch'].unique())
        
#         # Add slider
#         steps = []
#         for i, epoch in enumerate(df_latent_log['epoch'].unique()):
#             step = dict(
#                 method='restyle',
#                 args=['visible', [False] * len(trace_list)],
#                 label=f'{epoch}',
#             )
#             for j in range(n_crystal_types*2):
#                 step['args'][1][i*n_crystal_types*2 + j] = True
                
#             steps.append(step)  
        
#         sliders = [dict(
#             active=0,
#             currentvalue={"prefix": "Epoch: "},
#             pad={"t": 50},
#             steps=steps,
#         )]
        
#         fig.update_layout(
#             sliders=sliders,
#             width=1200,
#             height=800,
#             )
        
#         # Set axis labels
#         fig.update_xaxes(title_text='PC 1')
#         fig.update_yaxes(title_text='PC 2')
        
#         fig.write_html(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/latent_space_samples.html')
    

In [16]:
# Plot the latent space at specific epochs
epoch_to_plot = 0

if df_latent_log is not None:
    plt.figure(figsize=(10, 8))
    df_epoch = df_latent_log[df_latent_log['epoch'] == epoch_to_plot]
    df_epoch_samples = df_log_samples[df_log_samples['epoch'] == epoch_to_plot]
    
    if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
        sns.scatterplot(data=df_epoch_samples, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20', alpha=0.5, legend=False)
        sns.scatterplot(data=df_epoch, x='ls1', y='ls2', hue='crystal_type', style='crystal_type', s=100, palette='tab20', edgecolor='black')
        
        plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
        plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
        
        plt.xlim(df_log_samples['ls1'].min(), df_log_samples['ls1'].max())
        plt.ylim(df_log_samples['ls2'].min(), df_log_samples['ls2'].max())  
    else:
        sns.scatterplot(data=df_epoch_samples, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20', alpha=0.5, legend=False)
        sns.scatterplot(data=df_epoch, x='pc1', y='pc2', hue='crystal_type', style='crystal_type', s=100, palette='tab20', edgecolor='black')
        
        plt.xlabel('PC 1', fontsize=14, fontweight='bold')
        plt.ylabel('PC 2', fontsize=14, fontweight='bold')
        
        plt.xlim(df_log_samples['pc1'].min(), df_log_samples['pc1'].max())
        plt.ylim(df_log_samples['pc2'].min(), df_log_samples['pc2'].max())
    
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    # plt.axis('equal')
    plt.tight_layout()
    plt.show()

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_epoch_{epoch_to_plot}.pdf', dpi=300)

In [17]:
# Make a combined figure of the latent space at different epochs
epochs_to_plot = [0, 10, 100, 350, 690, 890]
subplot_rows = 2
subplot_cols = 3
figsize = (10, 6.5)

if df_latent_log is not None:
    fig, ax = plt.subplots(subplot_rows, subplot_cols, figsize=figsize, sharex=True, sharey=True)
    ax = ax.flatten()
    
    if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
        xlim_min = df_log_samples['ls1'].min()
        xlim_max = df_log_samples['ls1'].max()
        ylim_min = df_log_samples['ls2'].min()
        ylim_max = df_log_samples['ls2'].max()
    else:
        xlim_min = df_log_samples['pc1'].min()
        xlim_max = df_log_samples['pc1'].max()
        ylim_min = df_log_samples['pc2'].min()
        ylim_max = df_log_samples['pc2'].max()
    
    for i, epoch in enumerate(epochs_to_plot):
        df_epoch = df_latent_log[df_latent_log['epoch'] == epoch]
        df_epoch_samples = df_log_samples[df_log_samples['epoch'] == epoch]
        
        if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
            sns.histplot(data=df_epoch_samples, x='ls1', y='ls2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, legend=False, ax=ax[i])
            sns.scatterplot(data=df_epoch, x='ls1', y='ls2', hue='crystal_type', style='crystal_type', s=50, palette='tab20', edgecolor='black', ax=ax[i])
            
            ax[i].set_xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
            ax[i].set_ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
            
            ax[i].set_xlim(xlim_min, xlim_max)
            ax[i].set_ylim(ylim_min, ylim_max)
        else:
            sns.histplot(data=df_epoch_samples, x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, legend=False, ax=ax[i])
            sns.scatterplot(data=df_epoch, x='pc1', y='pc2', hue='crystal_type', style='crystal_type', s=50, palette='tab20', edgecolor='black', ax=ax[i])
            
            ax[i].set_xlabel('PC 1', fontsize=14, fontweight='bold')
            ax[i].set_ylabel('PC 2', fontsize=14, fontweight='bold')
            
            ax[i].set_xlim(xlim_min, xlim_max)
            ax[i].set_ylim(ylim_min, ylim_max)
        if i == 0:
            handles, labels = ax[i].get_legend_handles_labels()
            ax[i].legend(loc='lower center', bbox_to_anchor=(subplot_cols/2, 1), ncol=4)
        else:
            ax[i].legend([],[], frameon=False)
        ax[i].set_title(f' Epoch {epoch}', fontsize=12, fontweight='bold', y=1.0, pad=-15, loc='left')
    
    # Remove empty axes
    for i in range(len(epochs_to_plot), subplot_rows*subplot_cols):
        fig.delaxes(ax[i])
        
    # fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, 3), ncols=5)
    
    plt.tight_layout()
    
    # Remove whitespace
    plt.subplots_adjust(wspace=0.02, hspace=0.02)
    
    plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_epochs.pdf', dpi=300)

# Experimental data

## Code

In [18]:
data_folder = './data/Experimental/'

# Load experimental data
data_paths = [str(p) for p in pathlib.Path(data_folder).rglob('*.gr')]

data_filepath = []
data_composition_string = []
data_composition = []
data_pdf = []
for data_path in data_paths:
    with open(data_path, 'r') as f:
        # Load data
        line_counter = 0
        for line in f:
            if line.startswith('composition'):
                composition = ''.join(line.split(' ')[2:])
            if line.startswith('0'):
                header_line = line_counter
                break
            line_counter += 1
    # Remove line breaks
    composition = composition.replace('\n', '')
    
    # Save composition string
    composition_string = deepcopy(composition)

    # Remove stochiometry from composition
    composition = re.sub(r'[0-9\.]+', '', composition)

    # # Split string on capital letters
    composition = re.findall('[A-Z][^A-Z]*', composition)

    # Translate composition to atom numbers
    composition = Atoms(symbols=composition).get_atomic_numbers()
    
    composition_onehot = np.zeros(119)
    composition_onehot[composition] = 1
    
    
    # Load data
    data = pd.read_csv(data_path, sep=' ', skiprows=header_line, names=['r [Å]', 'G(r) [Å⁻²]'])
    
    data_r = np.arange(0,60,0.01)
    data_Gr = np.interp(data_r, data['r [Å]'], data['G(r) [Å⁻²]'], left=0, right=0)
    data_Gr = data_Gr / np.amax(data_Gr)
    
    data_filepath.append(data_path)
    data_composition_string.append(composition_string)
    data_composition.append(composition_onehot)
    data_pdf.append(data_Gr)

# Convert to tensors
data_composition = torch.tensor(data_composition, dtype=torch.long)
data_pdf = torch.tensor(data_pdf, dtype=torch.float32)
data_composition_string_index = torch.tensor(np.arange(len(data_composition_string)))
data_filepath_index = torch.tensor(np.arange(len(data_filepath)))

exp_data = TensorDataset(data_pdf, data_composition, data_composition_string_index, data_filepath_index)

# Dataloader
exp_loader = DataLoader(exp_data, batch_size=10, shuffle=False)

# Results dict
exp_results = {
    'composition': [],
    'pdf': [],
    'prior_mean': [],
    'prior_log_std': [],
    'z_sample': [],
    'cell_parameters': [],
    'cell_positions': [],
    'cell_atoms': [],
}

In [19]:
# Inference
model.eval()
for batch in tqdm(exp_loader, desc='Inference', disable=setup_json['disable_tqdm']):
    this_batch_size = len(batch[0])
    pdf, composition, composition_string_index, filepath_index = batch
    pdf = pdf.unsqueeze(-1).to(device)
    composition = composition.float().to(device)
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms, prior_mean, prior_log_std, z_sample = model.predict(
            pdf, 
            composition,
        )
    
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store composition
    for index in composition_string_index:
        exp_results['composition'].append(data_composition_string[index])
    
    # Store PDF
    exp_results['pdf'].extend(pdf.cpu().tolist())
    
    # Store latent representation
    exp_results['prior_mean'].extend(prior_mean.cpu().tolist())
    exp_results['prior_log_std'].extend(prior_log_std.cpu().tolist())
    exp_results['z_sample'].extend(z_sample.cpu().tolist())
    
    # Store predictions
    exp_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    exp_results['cell_positions'].extend(cell_positions.cpu().tolist())
    exp_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # Create CIF files
    for batch_index in range(this_batch_size):
        # Prediction
        try:
            create_cif(
                cell_params = cell_parameters[batch_index].detach().cpu().numpy(),
                cell_positions = cell_positions[batch_index].detach().cpu().numpy(),
                cell_atoms = cell_atoms[batch_index].detach().cpu().numpy(),
                filename = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/experimental_predictions/{data_filepath[data_filepath_index[batch_index]].split("/")[-1].split(".")[0]}',
                prediction=True,
                composition=data_composition_string[composition_string_index[batch_index]],
                simplified_atom_identities=setup_json['training']['simplified_atom_identities'],
            )
        except:
            print(f'Failed to create CIF file for prediction of {data_composition_string[composition_string_index[batch_index]]}.')
    
df_exp_results = pd.DataFrame(exp_results)
if len(df_exp_results['prior_mean'].iloc[0]) == 2:
    df_exp_results[['ls1', 'ls2']] = df_exp_results['prior_mean'].apply(pd.Series)
else:
    # Perform PCA
    df_exp_results[['pc1', 'pc2']] = pca.transform(np.array(df_exp_results['prior_mean'].values.tolist()))
df_exp_results.head()

# Drop IrO2
# df_exp_results = df_exp_results[df_exp_results['composition'] != 'IrO2']

,composition,pdf,prior_mean,prior_log_std,z_sample,cell_parameters,cell_positions,cell_atoms,ls1,ls2
0,IrO2,"[[0.0], [0.0007761651650071144], [0.0015256106...","[1.1934536695480347, 0.23632219433784485]","[0.060190349817276, 0.03700313717126846]","[1.250440001487732, 0.28758931159973145]","[8.58452320098877, 8.58427906036377, 8.5729961...","[[0.13279999792575836, 0.13634000718593597, 0....","[2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, ...",1.193454,0.236322
1,IrO2,"[[0.0], [0.0005720379413105547], [0.0011345194...","[1.2051013708114624, 0.23151865601539612]","[0.06218346953392029, 0.041738253086805344]","[1.156711459159851, 0.27231937646865845]","[8.636634826660156, 8.625781059265137, 8.63291...","[[0.11437000334262848, 0.14799000322818756, 0....","[2, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, ...",1.205101,0.231519
2,IrO2,"[[0.0], [-0.000292735145194456], [-0.000565890...","[1.1779347658157349, 0.23790103197097778]","[0.05904670059680939, 0.03361498937010765]","[1.23063325881958, 0.228692427277565]","[8.47106647491455, 8.453723907470703, 8.471536...","[[0.08686999976634979, 0.0925000011920929, 0.1...","[1, 2, 1, 1, 1, 1, 1, 2, 2, 1, 1, 1, 1, 1, 1, ...",1.177935,0.237901
3,CeO2,"[[0.0], [-0.000268803967628628], [-0.000530091...","[-0.9076144695281982, 0.29405903816223145]","[0.04237419366836548, 0.058713436126708984]","[-0.9610502123832703, 0.3708910346031189]","[10.971277236938477, 10.971197128295898, 10.97...","[[-0.0142000000923872, -0.0023499999660998583,...","[1, 2, 1, 1, 1, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2, ...",-0.907614,0.294059
4,CeO2,"[[0.0], [-0.0003009798820130527], [-0.00059319...","[-0.9106936454772949, 0.2927975058555603]","[0.042217593640089035, 0.05621759220957756]","[-0.8781991004943848, 0.3701167106628418]","[10.86315631866455, 10.885778427124023, 10.910...","[[-0.03757999837398529, -0.01761000044643879, ...","[1, 2, 1, 1, 1, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2, ...",-0.910694,0.292798


In [20]:
df_exp_results['ground_truth_crystal_type'] = None
df_exp_results['ground_truth_crystal_type'].iloc[0:3] = 'Rutile'
df_exp_results['ground_truth_crystal_type'].iloc[3:6] = 'Fluorite'
df_exp_results['ground_truth_crystal_type'].iloc[6:11] = 'Spinel'

## Plot

In [21]:
# Plot latent space placement of experimental data
if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=df_rec, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    sns.scatterplot(data=df_exp_results, x='ls1', y='ls2', hue='composition', style='ground_truth_crystal_type', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=df_rec, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    sns.scatterplot(data=df_exp_results, x='pc1', y='pc2', c='black', style='composition', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_experimental_samples.pdf', dpi=300)

In [22]:
# Plot the experimental data
# index_list = [0,1,2] # Rutile samples
index_list = [3, 4, 5] # Fluorite samples
# index_list = list(range(6,11)) # Spinel samples

plt.figure(figsize=(8, 6))
for index in index_list:
    plt.plot(np.arange(0,60,0.01), np.array(df_exp_results['pdf'].iloc[index]) + (index * 1.25), label=df_exp_results['composition'].iloc[index])
plt.xlabel('r [Å]', fontsize=14, fontweight='bold')
plt.ylabel('G(r) [A.U.]', fontsize=14, fontweight='bold')
plt.yticks([])
# plt.legend(ncols=2, loc='lower center', bbox_to_anchor=(0.5, 1.01))
plt.legend(ncols=3, loc='lower center', bbox_to_anchor=(0.5, 1.01))
# plt.title(f'{df_exp_results["composition"].iloc[index]}', fontsize=20, fontweight='bold')
plt.tight_layout()
plt.show()

# plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/experimental_data_Rutile.pdf', dpi=300)
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/experimental_data_Fluorite.pdf', dpi=300)
# plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/experimental_data_Spinel.pdf', dpi=300)

# Interpolate in latent space

## Code

In [23]:
# Create latent samples for interpolation between two points
n_latent_samples = 10

# Define two latent points from pca space
# latent_point_1 = (24, 1) # ex1 (9, -13)
# latent_point_2 = (24, -15) # ex1 (24,-9)

# Define n latent points from latent space
crystal_types = ['RockSalt', 'Spinel', 'ZincBlende'] #['CadmiumIodide', 'CadmiumChloride'] #['RockSalt', 'Spinel', 'ZincBlende']
indices = [0, 2, 4] #[14,3] #[0, 4, 0]
point_indices = [df_rec[(df_rec['crystalType'] == crystal_type)].index[index] for crystal_type, index in zip(crystal_types, indices)]

latent_points = [np.array(df_rec['latent_space_mean'].iloc[point_index]) for point_index in point_indices]

# Interpolation mode flag (True for PCA. False for full latent space)
pca_inter = False

# Define dummy composition
comp = ['Fe', 'O']
comp = Atoms(symbols=comp).get_atomic_numbers()
comp_onehot = torch.zeros(119)
comp_onehot[comp] = 1

# Latent samples are n_latent_samples evenly spaced points between the two points
if pca_inter:
    for i in range(len(latent_points)-1):
        if i == 0:
            interp_pca = np.array([np.linspace(latent_points[i][j], latent_points[i+1][j], n_latent_samples) for j in range(len(latent_points[i]))]).T
        else:
            interp_pca = np.concatenate([interp_pca, np.array([np.linspace(latent_points[i][j], latent_points[i+1][j], n_latent_samples) for j in range(len(latent_points[i]))]).T], axis=0)
    
    if len(df_rec['latent_space_mean'].iloc[0]) > 2:
        interp_latent = pca.inverse_transform(interp_pca)
    else:
        interp_latent = interp_pca
    interp_latent = torch.tensor(interp_latent).float()
else:
    # Transform back to latent dimensions if latent space is more than 2D
    if (len(df_rec['latent_space_mean'].iloc[0]) > 2) and (len(latent_points[0]) == 2):
        latent_points = pca.inverse_transform(latent_points)
    
    for i in range(len(latent_points)-1):
        if i == 0:
            interp_latent = np.array([np.linspace(latent_points[i][j], latent_points[i+1][j], n_latent_samples) for j in range(len(latent_points[i]))]).T
        else:
            interp_latent = np.concatenate([interp_latent, np.array([np.linspace(latent_points[i][j], latent_points[i+1][j], n_latent_samples) for j in range(len(latent_points[i]))]).T], axis=0)
        
    interp_latent = torch.tensor(interp_latent).float()

n_interp_points = interp_latent.shape[0]
interp_index = torch.tensor([i for i in range(n_interp_points)])

# Create dataset
interp_dataset = TensorDataset(interp_latent, interp_index, comp_onehot.repeat(n_interp_points, 1))

# Dataloader
interp_loader = DataLoader(interp_dataset, batch_size=10, shuffle=False)

# Results dict
interp_results = {
    'name': [],
    'latent_point': [],
    'cell_parameters': [],
    'cell_positions': [],
    'cell_atoms': [],
}

In [24]:
# Inference
model.eval()
for batch in tqdm(interp_loader, desc='Interpolating', disable=setup_json['disable_tqdm']):
    this_batch_size = len(batch[0])
    interp_point, interp_index, composition = batch
    interp_point = interp_point.to(device)
    interp_index = interp_index.to(device)
    composition = composition.to(device)
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms = model.decode(
            interp_point, 
            composition=composition,
        )
    
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store latent points
    interp_results['name'].extend([f'sample_{interp_index[batch_index]}' for batch_index in range(this_batch_size)])
    interp_results['latent_point'].extend(interp_point.cpu().tolist())
    
    # Store predictions
    interp_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    interp_results['cell_positions'].extend(cell_positions.cpu().tolist())
    interp_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # Create CIF files
    for batch_index in range(this_batch_size):
        # Prediction
        try:
            create_cif(
                cell_params = cell_parameters[batch_index].detach().cpu().numpy(),
                cell_positions = cell_positions[batch_index].detach().cpu().numpy(),
                cell_atoms = cell_atoms[batch_index].detach().cpu().numpy(),
                filename = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/interpolation_predictions/sample{interp_index[batch_index]}',
                prediction=True,
                composition=None,
                simplified_atom_identities=setup_json['training']['simplified_atom_identities'],
            )
        except:
            print(f'Failed to create CIF file for prediction of sample {interp_index[batch_index]}.')
    

In [25]:
df_interp = pd.DataFrame(interp_results)
if len(df_interp['latent_point'].iloc[0]) == 2:
    df_interp[['ls1', 'ls2']] = df_interp['latent_point'].apply(pd.Series)
else:
    # Perform PCA
    df_interp[['pc1', 'pc2']] = pca.transform(np.array(df_interp['latent_point'].values.tolist()))
df_interp.head()

,name,latent_point,cell_parameters,cell_positions,cell_atoms,ls1,ls2
0,sample_0,"[0.93146812915802, 0.40995854139328003]","[8.85799789428711, 8.876039505004883, 8.866258...","[[0.31073999404907227, 0.2888999879360199, 0.3...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0.931468,0.409959
1,sample_1,"[0.9357137680053711, 0.35366204380989075]","[9.015323638916016, 8.98257064819336, 8.979502...","[[0.16227999329566956, 0.15857000648975372, 0....","[1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 1, 1, 1, 1, 2, ...",0.935714,0.353662
2,sample_2,"[0.9399594068527222, 0.29736554622650146]","[8.861523628234863, 8.856945991516113, 8.85755...","[[0.11677999794483185, 0.13194000720977783, 0....","[2, 1, 1, 1, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, ...",0.939959,0.297366
3,sample_3,"[0.9442050457000732, 0.24106904864311218]","[8.640606880187988, 8.632013320922852, 8.64761...","[[0.08963000029325485, 0.09544000029563904, 0....","[1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0.944205,0.241069
4,sample_4,"[0.9484506845474243, 0.1847725510597229]","[8.531994819641113, 8.548157691955566, 8.53942...","[[0.043209999799728394, 0.04764999821782112, 0...","[1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 1, 1, ...",0.948451,0.184773


## Plot

In [26]:
# Plot interpolation results in latent space
if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df_rec, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    sns.scatterplot(data=df_interp, x='ls1', y='ls2', color='black', marker='o', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df_rec, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    sns.scatterplot(data=df_interp, x='pc1', y='pc2', hue='name', palette='viridis', marker='o', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_interpolation_samples.pdf', dpi=300)

In [27]:
# Plot the cell parameters of the interpolation with angles and lengths on two subplots (one for lengths and one for angles) that share the x-axis
df_cell_params = pd.DataFrame(df_interp['cell_parameters'].tolist(), columns=['a', 'b', 'c', 'alpha', 'beta', 'gamma'])
df_cell_params['name'] = df_interp['name']

fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Lengths
sns.lineplot(data=df_cell_params[['a', 'b', 'c']], ax=ax0)
ax0.set_ylabel('Length [Å]', fontsize=14, fontweight='bold')

# Angles
sns.lineplot(data=df_cell_params[['alpha', 'beta', 'gamma']], ax=ax1)
ax1.set_ylabel('Angle [°]', fontsize=14, fontweight='bold')
ax1.set_xlabel('Interpolation sample #', fontsize=14, fontweight='bold')
# y-axis label and ticks on the right side
ax1.yaxis.tick_right()
ax1.yaxis.set_label_position("right")

# Add vertical lines for the interpolation outside clusters and color between them
line_1 = 1.5
line_2 = 6.5
line_3 = 12.5
line_4 = 17.5
ax0.axvline(x=line_1, color='red', linestyle='--')
ax0.axvline(x=line_2, color='red', linestyle='--')
ax0.axvspan(line_1, line_2, color='red', alpha=0.1)

ax0.axvline(x=line_3, color='red', linestyle='--')
ax0.axvline(x=line_4, color='red', linestyle='--')
ax0.axvspan(line_3, line_4, color='red', alpha=0.1)

ax1.axvline(x=line_1, color='red', linestyle='--')
ax1.axvline(x=line_2, color='red', linestyle='--')
ax1.axvspan(line_1, line_2, color='red', alpha=0.1)

ax1.axvline(x=line_3, color='red', linestyle='--')
ax1.axvline(x=line_4, color='red', linestyle='--')
ax1.axvspan(line_3, line_4, color='red', alpha=0.1)

plt.xlim(0, n_interp_points-1)

# Legends
ax0.legend(loc='lower center' , bbox_to_anchor=(0.5, 0.05), ncol=3)
ax1.legend(loc='upper center' , bbox_to_anchor=(0.5, 0.95), ncol=3)

# Make x-ticks integers
plt.xticks(np.arange(0, n_interp_points, 1))

fig.tight_layout()

# Remove space between subplots
plt.subplots_adjust(hspace=0)

plt.show()

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/interpolation_cell_parameters.pdf', dpi=300)

# Sample same distribution multiple times

## Code

In [28]:
# Number of samples to generate
n_samples = 10

# Latent distribution to sample from
dist_crystal_type = 'Spinel'
crystal_type_index = 3
dist_index = df_rec[(df_rec['crystalType'] == dist_crystal_type)].index[crystal_type_index]
dist_mean = df_rec.latent_space_mean.iloc[dist_index]
dist_std = df_rec.latent_space_std.iloc[dist_index]

# Latent distribution from experimental data
# exp_index = 3
# dist_mean = df_exp_results.prior_mean.iloc[exp_index]
# dist_std = df_exp_results.prior_log_std.iloc[exp_index]

latent_dist = torch.distributions.Normal(loc=torch.tensor(dist_mean).float(), scale=torch.tensor(dist_std).float())

# Sample from latent distribution
latent_mean = latent_dist.mean
latent_samples = latent_dist.sample((n_samples,))
# Concatenate with latent mean
latent_samples = torch.cat((latent_mean.unsqueeze(0), latent_samples), dim=0)
sample_index = torch.tensor([i for i in range(n_samples+1)])

# Define dummy composition
comp = ['Fe', 'O']
comp = Atoms(symbols=comp).get_atomic_numbers()
comp_onehot = torch.zeros(119)
comp_onehot[comp] = 1

# Define dataset
sample_dataset = TensorDataset(latent_samples, sample_index, comp_onehot.repeat(n_samples+1, 1))

# Dataloader
sample_loader = DataLoader(sample_dataset, batch_size=10, shuffle=False)

# Results dict
sample_results = {
    'name': [],
    'latent_point': [],
    'cell_parameters': [],
    'cell_positions': [],
    'cell_atoms': [],
}

In [29]:
# Inference
model.eval()
for batch in tqdm(sample_loader, desc='Sampling', disable=setup_json['disable_tqdm']):
    this_batch_size = len(batch[0])
    sample_point, sample_index, composition = batch
    sample_point = sample_point.to(device)
    sample_index = sample_index.to(device)
    composition = composition.to(device)
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms = model.decode(
            sample_point, 
            composition=composition,
        )
    
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store latent points
    sample_results['name'].extend([f'sample_{sample_index[batch_index]}' for batch_index in range(this_batch_size)])
    sample_results['latent_point'].extend(sample_point.cpu().tolist())
    
    # Store predictions
    sample_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    sample_results['cell_positions'].extend(cell_positions.cpu().tolist())
    sample_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # Create CIF files
    for batch_index in range(this_batch_size):
        # Prediction
        try:
            create_cif(
                cell_params = cell_parameters[batch_index].detach().cpu().numpy(),
                cell_positions = cell_positions[batch_index].detach().cpu().numpy(),
                cell_atoms = cell_atoms[batch_index].detach().cpu().numpy(),
                filename = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/sample_predictions/sample{sample_index[batch_index]}',
                prediction=True,
                composition=None,
                simplified_atom_identities=setup_json['training']['simplified_atom_identities'],
            )
        except:
            print(f'Failed to create CIF file for prediction of sample {sample_index[batch_index]}.')

In [30]:
df_sample = pd.DataFrame(sample_results)
if len(df_sample['latent_point'].iloc[0]) == 2:
    df_sample[['ls1', 'ls2']] = df_sample['latent_point'].apply(pd.Series)
else:
    # Perform PCA
    df_sample[['pc1', 'pc2']] = pca.transform(np.array(df_sample['latent_point'].values.tolist()))
    
df_sample['name'][df_sample['name'] == 'sample_0'] = 'Dist. mean'
df_sample.head()

,name,latent_point,cell_parameters,cell_positions,cell_atoms,ls1,ls2
0,Dist. mean,"[-1.0716575384140015, -0.17223891615867615]","[10.540711402893066, 10.551360130310059, 10.55...","[[0.02741999924182892, 0.02751999907195568, 0....","[1, 2, 2, 2, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 2, ...",-1.071658,-0.172239
1,sample_1,"[-1.0567128658294678, -0.1519898772239685]","[10.509432792663574, 10.541313171386719, 10.53...","[[0.02442000061273575, 0.024380000308156013, 0...","[1, 2, 2, 2, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 2, ...",-1.056713,-0.151990
2,sample_2,"[-1.0691108703613281, -0.16723313927650452]","[10.534058570861816, 10.548975944519043, 10.55...","[[0.0266100000590086, 0.027070000767707825, 0....","[1, 2, 2, 2, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 2, ...",-1.069111,-0.167233
3,sample_3,"[-1.0636883974075317, -0.19236302375793457]","[10.543853759765625, 10.539608001708984, 10.55...","[[0.026149999350309372, 0.026660000905394554, ...","[1, 2, 2, 2, 1, 1, 1, 2, 1, 1, 1, 2, 1, 1, 2, ...",-1.063688,-0.192363
4,sample_4,"[-1.0756886005401611, -0.20342734456062317]","[10.571737289428711, 10.563969612121582, 10.57...","[[0.024890000000596046, 0.023679999634623528, ...","[1, 2, 2, 2, 1, 1, 1, 2, 1, 1, 1, 2, 1, 1, 1, ...",-1.075689,-0.203427


## Plot

In [31]:
# Plot sampling results in latent space
if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=df_rec, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    sns.scatterplot(data=df_sample[df_sample['name'] != 'Dist. mean'], x='ls1', y='ls2', color='black', marker='.', s=100, label='Samples')
    sns.scatterplot(data=df_sample[df_sample['name'] == 'Dist. mean'], x='ls1', y='ls2', color='red', marker='.', s=200, label='Dist. mean')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('LS dim 1')
    plt.ylabel('LS dim 2')
    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=df_rec, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    # sns.scatterplot(data=df_sample, x='pc1', y='pc2', hue='name', palette='viridis', marker='o', s=100)
    sns.scatterplot(data=df_sample[df_sample['name'] != 'Dist. mean'], x='pc1', y='pc2', color='black', marker='.', s=100, label='Samples')
    sns.scatterplot(data=df_sample[df_sample['name'] == 'Dist. mean'], x='pc1', y='pc2', color='red', marker='.', s=200, label='Dist. mean')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1')
    plt.ylabel('PC 2')
    plt.tight_layout()
    plt.show()

# Sample every latent distrubution and calculate loss

In [32]:
# Number of samples per point
n_samples = 100 #1000
test_data = 'validation'

# Seed for reproducibility
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# Sample from each latent distribution
data_names = []
data_samples = []
data_sample_type = []
data_crystal_types = []
ground_truth_cell_parameters = []
ground_truth_cell_positions = []
ground_truth_cell_atoms = []
for data_index in range(len(df_rec)):
    latent_mean = df_rec['latent_space_mean'].iloc[data_index]
    latent_std = df_rec['latent_space_std'].iloc[data_index]
    latent_dist = torch.distributions.Normal(loc=torch.tensor(latent_mean).float(), scale=torch.tensor(latent_std).float())
    latent_samples = latent_dist.sample((n_samples,))
    
    data_samples.append(torch.tensor(latent_mean))
    data_samples.extend([latent_sample for latent_sample in latent_samples])
    
    data_names.append(f'{data_index}')
    data_names.extend([f'{data_index}_{i}' for i in range(n_samples)])
    
    data_sample_type.append('mean')
    data_sample_type.extend(['sample' for i in range(n_samples)])
    
    data_crystal_types.extend([df_rec['crystalType'].iloc[data_index] for i in range(n_samples+1)])
    
    # Ground truth
    ground_truth_cell_parameters.extend([df_rec['cell_parameters'].iloc[data_index]]*(n_samples+1))
    ground_truth_cell_positions.extend([df_rec['cell_positions'].iloc[data_index]]*(n_samples+1))
    ground_truth_cell_atoms.extend([df_rec['cell_atoms'].iloc[data_index]]*(n_samples+1))
    
data_samples = torch.stack(data_samples)
ground_truth_cell_parameters = torch.tensor(ground_truth_cell_parameters)
ground_truth_cell_positions = torch.tensor(ground_truth_cell_positions)
ground_truth_cell_atoms = torch.tensor(ground_truth_cell_atoms)

# Define dummy composition
comp = ['Fe', 'O']
comp = Atoms(symbols=comp).get_atomic_numbers()
comp_onehot = torch.zeros(119)
comp_onehot[comp] = 1

data_index = torch.tensor(np.arange(len(data_names)))

# Define dataset
sample_dataset = TensorDataset(data_samples, data_index, comp_onehot.repeat(len(data_names), 1), ground_truth_cell_parameters, ground_truth_cell_positions, ground_truth_cell_atoms)

# Dataloader
sample_loader = DataLoader(sample_dataset, batch_size=10, shuffle=False)

# Results dict
sample_results = {
    'name': [],
    'sample_type': [],
    'crystalType': [],
    'latent_point': [],
    'cell_parameters': [],
    'cell_positions': [],
    'cell_atoms': [],
    'reconstruction_loss': [],
}
    

In [33]:
# Loss functions
loss_fn_cell_parameters = torch.nn.MSELoss()
# loss_fn_cell_positions = torch.nn.MSELoss()
# loss_fn_cell_atoms = torch.nn.CrossEntropyLoss()
# loss_fn_kld = torch.nn.KLDivLoss()
loss_fn_cell_positions = weighted_MSELoss()
loss_fn_cell_atoms = weighted_CrossEntropyLoss()
loss_fn_latent_mean = torch.nn.MSELoss()
loss_fn_latent_std = torch.nn.MSELoss()

In [34]:
# Inference
model.eval()
for batch in tqdm(sample_loader, desc='Sampling', disable=setup_json['disable_tqdm']):
    this_batch_size = len(batch[0])
    sample_point, sample_index, composition, cell_parameters_true, cell_positions_true, cell_atoms_true = batch
    sample_point = sample_point.to(device)
    sample_indeces = sample_index.to(device)
    composition = composition.to(device)
    cell_parameters_true = cell_parameters_true.to(device)
    cell_positions_true = cell_positions_true.to(device)
    cell_atoms_true = cell_atoms_true.to(device)
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms = model.decode(
            sample_point, 
            composition=composition,
        )
    
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store latent points
    for sample_index in sample_indeces:
        sample_results['name'].append(data_names[sample_index])
        sample_results['sample_type'].append(data_sample_type[sample_index])
        sample_results['crystalType'].append(data_crystal_types[sample_index])
    
    sample_results['latent_point'].extend(sample_point.cpu().tolist())
    
    # Store predictions
    sample_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    sample_results['cell_positions'].extend(cell_positions.cpu().tolist())
    sample_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # Make loss weights
    cell_positions_weights = torch.where(cell_positions_true != -1, 1, 0).float().to(device)
    cell_atoms_weights = torch.where(cell_atoms_true != 0, 1, 0.1).float().to(device)

    # Simplify atom identities
    if setup_json['training']['simplified_atom_identities']:
        # Map atom number 0 to logit 0 (No atom)
        cell_atoms_true = torch.where(cell_atoms_true == 0, 0, cell_atoms_true)
        # Map atom numbers of ligands to logit 1 (Ligand) # [1, 6, 7, 8, 9, 15, 16, 17, 34, 35, 53]
        for ligand in setup_json['training']['ligands']:
            cell_atoms_true = torch.where(cell_atoms_true == ligand, 1, cell_atoms_true)
        # Map all other atom numbers to logit 2 (Metal)
        cell_atoms_true = torch.where(cell_atoms_true >= 2, 2, cell_atoms_true)
    
    # Calculate reconstruction loss for each sample
    for batch_index in range(this_batch_size):
        loss_cell_parameters = loss_fn_cell_parameters(cell_parameters[batch_index], cell_parameters_true[batch_index])
        loss_cell_positions = loss_fn_cell_positions(cell_positions[batch_index], cell_positions_true[batch_index], cell_positions_weights[batch_index])
        loss_cell_atoms = loss_fn_cell_atoms(cell_atoms[batch_index], cell_atoms_true[batch_index], cell_atoms_weights[batch_index])
        
        loss_reconstruction = loss_cell_parameters + loss_cell_positions + loss_cell_atoms
        
        # Save loss
        sample_results['reconstruction_loss'].append(loss_reconstruction.item())
        
    # # Reconstruction loss
    # loss_cell_parameters = loss_fn_cell_parameters(cell_parameters, cell_parameters_true) 
    
    # # loss_cell_positions = loss_fn_cell_positions(cell_positions, cell_positions_true) # Unweighted
    # loss_cell_positions = loss_fn_cell_positions(cell_positions, cell_positions_true, cell_positions_weights) # Weighted
    
    # # loss_cell_atoms = loss_fn_cell_atoms(cell_atoms, cell_atoms_true) # Unweighted
    # loss_cell_atoms = loss_fn_cell_atoms(cell_atoms, cell_atoms_true, cell_atoms_weights) # Weighted
    
    # loss_reconstruction = loss_cell_parameters + loss_cell_positions + loss_cell_atoms
    
    # # Save loss
    # sample_results['reconstruction_loss'].extend([loss_reconstruction.item()]*this_batch_size)


In [35]:
df_full_latent = pd.DataFrame(sample_results)
if len(df_full_latent['latent_point'].iloc[0]) == 2:
    df_full_latent[['ls1', 'ls2']] = df_full_latent['latent_point'].apply(pd.Series)
else:
    # Perform PCA
    df_full_latent[['pc1', 'pc2']] = pca.transform(np.array(df_full_latent['latent_point'].values.tolist())
)
df_full_latent.head()

,name,sample_type,crystalType,latent_point,cell_parameters,cell_positions,cell_atoms,reconstruction_loss,ls1,ls2
0,0,mean,Interpolated,"[1.3172607421875, 0.6281828880310059]","[8.618607521057129, 8.620262145996094, 8.60307...","[[0.03268999978899956, -0.022630000486969948, ...","[1, 2, 2, 2, 1, 1, 1, 2, 1, 1, 1, 2, 2, 2, 2, ...",0.250643,1.317261,0.628183
1,0_0,sample,Interpolated,"[1.479330062866211, 0.6590681076049805]","[8.53303337097168, 8.523818016052246, 8.520594...","[[0.019300000742077827, 0.005900000222027302, ...","[1, 2, 2, 2, 1, 1, 1, 2, 1, 1, 1, 2, 2, 2, 2, ...",1.874188,1.479330,0.659068
2,0_1,sample,Interpolated,"[1.393018364906311, 0.5844591856002808]","[8.594232559204102, 8.604634284973145, 8.59084...","[[0.007890000008046627, -0.0014900000533089042...","[1, 2, 2, 2, 1, 1, 1, 2, 1, 1, 1, 2, 2, 2, 2, ...",1.549136,1.393018,0.584459
3,0_2,sample,Interpolated,"[1.3743212223052979, 0.6025460958480835]","[8.595958709716797, 8.59953498840332, 8.590460...","[[0.0328500010073185, -0.004819999914616346, 0...","[1, 2, 2, 2, 1, 1, 1, 2, 1, 1, 1, 2, 2, 2, 2, ...",0.373542,1.374321,0.602546
4,0_3,sample,Interpolated,"[1.31363844871521, 0.5948600172996521]","[8.627466201782227, 8.629571914672852, 8.61041...","[[0.019910000264644623, -0.005570000037550926,...","[1, 2, 2, 2, 1, 1, 1, 2, 1, 1, 1, 2, 2, 2, 2, ...",1.259596,1.313638,0.594860


In [36]:
# Plot sampling results in latent space
if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75)
    # sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', style='crystalType', palette='tab20', s=50)    
    sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75)
    # sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', style='crystalType', palette='tab20', s=50)
    sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_sample_density.pdf', dpi=300)

In [37]:
# Plot latent space with surface showing the loss
from scipy.interpolate import griddata
import matplotlib.colors as colors

# Decide on contour column
contour_column = 'reconstruction_loss'

# Get min and max values
min_loss = df_full_latent[contour_column].min()
max_loss = df_full_latent[contour_column].max()

# Plot
if len(df_full_latent['latent_point'].iloc[0]) == 2:
    # Interpolate z values to a grid from points
    xi = np.linspace(df_full_latent['ls1'].min()-0.1, df_full_latent['ls1'].max()+0.1, 1000)
    yi = np.linspace(df_full_latent['ls2'].min()-0.1, df_full_latent['ls2'].max()+0.1, 1000)
    zi = griddata((df_full_latent['ls1'], df_full_latent['ls2']), df_full_latent[contour_column], (xi[None,:], yi[:,None]), method='linear', fill_value=np.nan)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xi, yi, zi, 200, cmap='viridis')#, norm=colors.LogNorm())
    sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='ls1', y='ls2', hue='crystalType', style='crystalType', palette='tab20', s=50)
    plt.colorbar()
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    
    # Interpolate z values to a grid from points
    xi = np.linspace(df_full_latent['pc1'].min()-0.1, df_full_latent['pc1'].max()+0.1, 1000)
    yi = np.linspace(df_full_latent['pc2'].min()-0.1, df_full_latent['pc2'].max()+0.1, 1000)
    zi = griddata((df_full_latent['pc1'], df_full_latent['pc2']), df_full_latent[contour_column], (xi[None,:], yi[:,None]), method='linear', fill_value=np.nan)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xi, yi, zi, 200, cmap='viridis')#, norm=colors.LogNorm())
    sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    plt.colorbar()
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_sample_loss.pdf', dpi=300)	

In [38]:
df_full_latent.head()

,name,sample_type,crystalType,latent_point,cell_parameters,cell_positions,cell_atoms,reconstruction_loss,ls1,ls2
0,0,mean,Interpolated,"[1.3172607421875, 0.6281828880310059]","[8.618607521057129, 8.620262145996094, 8.60307...","[[0.03268999978899956, -0.022630000486969948, ...","[1, 2, 2, 2, 1, 1, 1, 2, 1, 1, 1, 2, 2, 2, 2, ...",0.250643,1.317261,0.628183
1,0_0,sample,Interpolated,"[1.479330062866211, 0.6590681076049805]","[8.53303337097168, 8.523818016052246, 8.520594...","[[0.019300000742077827, 0.005900000222027302, ...","[1, 2, 2, 2, 1, 1, 1, 2, 1, 1, 1, 2, 2, 2, 2, ...",1.874188,1.479330,0.659068
2,0_1,sample,Interpolated,"[1.393018364906311, 0.5844591856002808]","[8.594232559204102, 8.604634284973145, 8.59084...","[[0.007890000008046627, -0.0014900000533089042...","[1, 2, 2, 2, 1, 1, 1, 2, 1, 1, 1, 2, 2, 2, 2, ...",1.549136,1.393018,0.584459
3,0_2,sample,Interpolated,"[1.3743212223052979, 0.6025460958480835]","[8.595958709716797, 8.59953498840332, 8.590460...","[[0.0328500010073185, -0.004819999914616346, 0...","[1, 2, 2, 2, 1, 1, 1, 2, 1, 1, 1, 2, 2, 2, 2, ...",0.373542,1.374321,0.602546
4,0_3,sample,Interpolated,"[1.31363844871521, 0.5948600172996521]","[8.627466201782227, 8.629571914672852, 8.61041...","[[0.019910000264644623, -0.005570000037550926,...","[1, 2, 2, 2, 1, 1, 1, 2, 1, 1, 1, 2, 2, 2, 2, ...",1.259596,1.313638,0.594860


In [39]:
# Plot all samples in 3d if latent space is 3D

# if len(df_rec['latent_space_mean'].iloc[0]) == 3:
    
#     df_crystal = df_full_latent.copy()
#     df_crystal[['ls1', 'ls2', 'ls3']] = df_crystal['latent_point'].apply(pd.Series)
    
#     fig = px.scatter_3d(df_crystal, x='ls1', y='ls2', z='ls3', color='crystalType', symbol='crystalType', hover_data=['name', 'ls1', 'ls2', 'ls3', 'reconstruction_loss', 'cell_parameters'], color_discrete_sequence=px.colors.qualitative.Dark24)
    
#     fig.update_layout(
#         scene = dict(
#             xaxis_title='LS dim 1',
#             yaxis_title='LS dim 2',
#             zaxis_title='LS dim 3',
#         ),
#         margin=dict(l=0, r=0, t=0, b=0),
#     )
    # fig.show()